In [ ]:
from nanover.openmm import OpenMMSimulation
from nanover.mdanalysis.converter import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

In [ ]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="MOVEABLE RESTRAINTS")
imd_runner.load(0)
imd = imd_runner.app_server.imd

In [ ]:
from nanover.jupyter import show_runner_controls

show_runner_controls(imd_runner)

In [ ]:
from nanover.imd.imd_state import ParticleInteraction
from ipywidgets import Output

def reset():
    with output:
        display("RESET")
    for atom in universe.atoms.indices:
        imd.insert_interaction("interaction.TEST" + str(atom), ParticleInteraction((0, 0, 0), scale=0, reset_velocities=True, particles=[atom]))
    simulation.advance_by_one_step()
    for atom in universe.atoms.indices:
        imd.remove_interaction("interaction.TEST" + str(atom))
    simulation.advance_by_one_step()


def on_interaction_started(*, key: str, interaction: ParticleInteraction):
    if "TEST" in key:
        return
    imd_runner.play()


def on_interaction_stopped(*, key: str, interaction: ParticleInteraction):
    if "TEST" in key:
        return
    if not imd.active_interactions:
        try:
            imd_runner.pause()
            reset()
        except Exception as e:
            with output:
                display(e)
        # imd_runner.pause()


imd_runner.app_server.imd.interaction_started.add_callback(on_interaction_started)
imd_runner.app_server.imd.interaction_stopped.add_callback(on_interaction_stopped)

output = Output()
output